# Project 2: retrieval and T5 scoring
Run on an A40 or A30 Jupyter session. Run the cells in order.

In [1]:
import glob, os, re, shutil, subprocess, sys

SCRATCH = os.environ.get("RCAC_SCRATCH") or os.path.expanduser("~")
os.environ["HF_HOME"] = f"{SCRATCH}/hf_cache"
os.environ["PYSERINI_CACHE"] = f"{SCRATCH}/pyserini_cache"
INDEX = "wikipedia-dpr-100w"

def sh(cmd):
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

os.makedirs("data", exist_ok=True)
if not os.path.exists("data/dev.jsonl"):
    src = [p for p in ["../lab3/data/dev.jsonl", "../lab2/data/dev.jsonl"] if os.path.exists(p)]
    if not src:
        raise FileNotFoundError("Put dev.jsonl in data/ or keep this folder next to lab3.")
    shutil.copy(src[0], "data/dev.jsonl")

RETR_ENV = f"{SCRATCH}/lab3_pyserini_env"
if not os.path.exists(f"{RETR_ENV}/bin/python"):
    sh(f"{sys.executable} -m venv {RETR_ENV}")
    sh(f"{RETR_ENV}/bin/pip install -q --upgrade pip")
    sh(f"{RETR_ENV}/bin/pip install -q pyserini install-jdk")

def java_major():
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except FileNotFoundError:
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0

if java_major() < 21:
    jdks = glob.glob(f"{SCRATCH}/jdk/jdk-21*")
    if not jdks:
        sh(f"{RETR_ENV}/bin/python -c \"import jdk; jdk.install('21', path='{SCRATCH}/jdk')\"")
        jdks = glob.glob(f"{SCRATCH}/jdk/jdk-21*")
    os.environ["JAVA_HOME"] = jdks[0]
    os.environ["PATH"] = f"{jdks[0]}/bin:" + os.environ["PATH"]
print("Java", java_major())

GEN_PY = None
for py in [f"{SCRATCH}/lab3_gen_env/bin/python", f"{SCRATCH}/lab2_env/bin/python", sys.executable]:
    if os.path.exists(py) and subprocess.run([py, "-c", "import torch, transformers"], capture_output=True).returncode == 0:
        GEN_PY = py
        break
if GEN_PY is None:
    GEN_ENV = f"{SCRATCH}/project2_env"
    sh(f"{sys.executable} -m venv {GEN_ENV}")
    sh(f"{GEN_ENV}/bin/pip install -q --upgrade pip")
    sh(f"{GEN_ENV}/bin/pip install -q torch transformers")
    GEN_PY = f"{GEN_ENV}/bin/python"
if subprocess.run([GEN_PY, "-c", "import sentencepiece"], capture_output=True).returncode != 0:
    sh(f"{GEN_PY} -m pip install -q sentencepiece")
print("Python:", GEN_PY)

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise RuntimeError("No GPU in this session. Start the Jupyter session on an A40 or A30.")
print("GPU:", gpu)


Java 21
Python: /scratch/scholar/chaud124/lab3_gen_env/bin/python
GPU: NVIDIA A40


## Step 1: top 15 BM25 passages for every dev question

In [2]:
sh(f"{RETR_ENV}/bin/python retrieval.py cache --index {INDEX} --threads 8")

1110 questions, 1110 to retrieve
256/1110
512/1110
768/1110
1024/1110
1110/1110


## Step 2: score all 15 passages per question with monoT5
Finished questions are skipped if you rerun this cell.

In [3]:
sh(f"{GEN_PY} retrieval.py score")

Loading weights: 100%|██████████| 511/511 [00:00<00:00, 2050.45it/s]


1110 questions, 1110 to score
50/1110  0.30s per question
100/1110  0.21s per question
150/1110  0.18s per question
200/1110  0.17s per question
250/1110  0.16s per question
300/1110  0.15s per question
350/1110  0.15s per question
400/1110  0.14s per question
450/1110  0.14s per question
500/1110  0.14s per question
550/1110  0.14s per question
600/1110  0.14s per question
650/1110  0.13s per question
700/1110  0.13s per question
750/1110  0.13s per question
800/1110  0.13s per question
850/1110  0.13s per question
900/1110  0.13s per question
950/1110  0.13s per question
1000/1110  0.13s per question
1050/1110  0.13s per question
1100/1110  0.13s per question


## Step 3: threshold and cap sweep

In [4]:
sh(f"{GEN_PY} retrieval.py sweep")
shutil.make_archive("project2_retrieval_results", "zip", ".", "outputs")
print("Done. Download project2_retrieval_results.zip.")


1110 questions, keep=2, fallback=top
 cap   thr  fallback%   docs  early%  gold_sel%  gold_bm25%
   5  0.10       2.34   5.00   97.66      40.86       36.67
   5  0.20       4.14   5.00   95.86      40.86       36.67
   5  0.30       5.68   5.00   94.32      40.86       36.67
   5  0.40       7.03   5.00   92.97      40.76       36.67
   5  0.50       9.10   5.00   90.90      40.76       36.67
   5  0.60      11.08   5.00   88.92      40.76       36.67
   5  0.70      13.33   5.00   86.67      40.67       36.67
   5  0.80      17.12   5.00   82.88      40.38       36.67
   5  0.90      25.41   5.00   74.59      40.48       36.67
  10  0.10       1.62   5.12   98.38      40.86       36.67
  10  0.20       2.61   5.21   97.39      40.95       36.67
  10  0.30       3.96   5.28   96.04      41.05       36.67
  10  0.40       5.41   5.35   94.59      40.95       36.67
  10  0.50       6.85   5.45   93.15      40.95       36.67
  10  0.60       8.74   5.55   91.26      40.95       36.67
  1